# Carrera de Data Analyst — Python: `lambda`, `map()` y `apply()`

Cuaderno práctico con los ejemplos explicados: funciones lambda, transformación de listas, uso con Pandas, clasificación de clientes y múltiples parámetros.

**Objetivo:** comprender qué hace cada herramienta y cuándo utilizarla en un análisis financiero.

## 1. Lambda: función breve y anónima

Sintaxis general:
```python
lambda argumento1, argumento2: expresion
```

Lambda devuelve automáticamente el resultado de la expresión; no necesita `return`.

In [ ]:
# Ejemplo equivalente usando def y lambda
def sumar_def(a, b):
    return a + b

sumar_lambda = lambda a, b: a + b

print(sumar_def(10, 5))
print(sumar_lambda(10, 5))

In [ ]:
# Otros ejemplos sencillos de lambda
duplicar = lambda x: x * 2
cuadrado = lambda x: x ** 2
convertir_a_millones = lambda x: x / 1_000_000

print(duplicar(8))
print(cuadrado(5))
print(convertir_a_millones(2_500_000))

## 2. `map()` de Python: transformar elementos

Sintaxis:
```python
map(funcion, iterable)
```

`map()` aplica la función a cada elemento. Devuelve un iterador; utilizamos `list()` cuando queremos materializar el resultado como lista.

In [ ]:
saldos = [1_000_000, 2_000_000, 3_000_000, 4_000_000]

resultado = map(lambda x: x / 1_000_000, saldos)
print(list(resultado))

## 3. `map()` con más de un parámetro e iterable

`map()` puede recibir varios iterables. La función debe aceptar la misma cantidad de argumentos. Los elementos se combinan por posición y el recorrido termina cuando se agota el iterable más corto.

In [ ]:
saldos = [1_000_000, 2_000_000, 3_000_000]
costos = [200_000, 300_000, 500_000]

margenes = list(
    map(lambda saldo, costo: saldo - costo, saldos, costos)
)

print(margenes)

## 4. Preparar una base de clientes con Pandas

En los siguientes ejemplos utilizaremos un DataFrame pequeño para simular una base de clientes de una entidad financiera.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Cliente": ["A", "B", "C", "D", "E"],
    "Segmento_Actual": ["Minorista", "Minorista", "Mayorista", "Mayorista", "Minorista"],
    "Saldo": [2_000_000, 15_000_000, 80_000_000, 120_000_000, 500_000]
})

display(df)

## 5. `.map()` de Pandas: transformar una columna

A diferencia del `map()` incorporado de Python, el método `.map()` de una Serie de Pandas devuelve otra Serie. No hace falta convertirla a lista.

In [ ]:
df["Saldo_Millones"] = df["Saldo"].map(
    lambda x: x / 1_000_000
)

display(df)

## 6. `.map()` + `lambda`: clasificación por saldo

La lógica es equivalente a un `CASE WHEN` de SQL:
- Menor a 5 millones: Bajo.
- Desde 5 millones y menor a 50 millones: Medio.
- 50 millones o más: Alto.

In [ ]:
df["Nivel_Saldo"] = df["Saldo"].map(
    lambda x: "Bajo" if x < 5_000_000
    else "Medio" if x < 50_000_000
    else "Alto"
)

display(df[["Cliente", "Saldo", "Nivel_Saldo"]])

In [ ]:
# Crear una prioridad comercial ilustrativa a partir de la categoría
df["Oportunidad"] = df["Nivel_Saldo"].map(
    lambda nivel: "Prioridad alta" if nivel == "Alto"
    else "Seguimiento" if nivel == "Medio"
    else "Baja prioridad"
)

display(df)

## 7. `.apply()` de Pandas: aplicar una función

`apply()` puede trabajar sobre una Serie o un DataFrame. En una Serie, recibe cada valor. En un DataFrame, `axis=1` permite trabajar fila por fila.

In [ ]:
# apply sobre una columna (Serie)
df["Categoria_apply"] = df["Saldo"].apply(
    lambda x: "Bajo" if x < 5_000_000
    else "Medio" if x < 50_000_000
    else "Alto"
)

display(df[["Cliente", "Saldo", "Categoria_apply"]])

## 8. `.apply()` con varias columnas: crecimiento del saldo

Con `axis=1`, la función recibe una fila completa. Podemos acceder a varias columnas mediante `fila["nombre_columna"]`.

In [ ]:
df["Saldo_Anterior"] = [4_000_000, 20_000_000, 70_000_000, 100_000_000, 1_000_000]

df["Crecimiento_%"] = df.apply(
    lambda fila: (fila["Saldo"] / fila["Saldo_Anterior"] - 1) * 100,
    axis=1
)

display(df[["Cliente", "Saldo_Anterior", "Saldo", "Crecimiento_%"]])

## 9. `apply()` con un argumento adicional (`args`)

También podemos definir una función normal y pasarle parámetros adicionales.

In [ ]:
def calcular_impuesto(valor, tasa):
    return valor * tasa

df["Impuesto_10"] = df["Saldo"].apply(
    calcular_impuesto,
    args=(0.10,)
)

display(df[["Cliente", "Saldo", "Impuesto_10"]])

## 10. Alternativa vectorizada: `numpy.select()`

Para clasificaciones sobre bases grandes, las operaciones vectorizadas suelen ser más eficientes que ejecutar una función Python fila por fila.

In [ ]:
import numpy as np

condiciones = [
    df["Saldo"] < 5_000_000,
    df["Saldo"] < 50_000_000
]

categorias = ["Bajo", "Medio"]

df["Nivel_Saldo_numpy"] = np.select(
    condiciones,
    categorias,
    default="Alto"
)

display(df[["Cliente", "Saldo", "Nivel_Saldo_numpy"]])

## 11. Resumen para recordar

| Herramienta | Idea principal |
|---|---|
| `lambda` | Define una función breve: qué operación realizar. |
| `map()` de Python | Aplica una función a elementos de uno o más iterables. |
| `.map()` de Pandas | Transforma los valores de una Serie. |
| `.apply()` de Pandas | Aplica una función a una Serie o a filas/columnas de un DataFrame. |
| `np.select()` | Clasifica datos mediante condiciones vectorizadas. |

**Nota de rendimiento:** para operaciones numéricas sencillas en Pandas, preferí operaciones vectorizadas cuando sea posible. `apply(axis=1)` puede ser más lento en DataFrames grandes.